# Prompt Ablation Study

**Project:** Vibe-Synth  
**Purpose:** Measure how individual prompt components affect generation quality.
Three ablations are tested:

1. **AST hint ablation** — does injecting the template chain hint improve compilation success rate?
2. **Few-shot ablation** — does including few-shot examples reduce self-correction rounds?
3. **Cache threshold sweep** — what similarity threshold maximises cache hit rate without degrading quality?

Each ablation runs a fixed set of Vibes with and without the component under test, then compares outcomes.

In [ ]:
# ---------------------------------------------------------------------------
# Dependencies
# ---------------------------------------------------------------------------
import sys
import json
import time
import asyncio
import copy
import httpx
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from pathlib import Path
from IPython.display import display

sys.path.insert(0, "..")

API_BASE = "http://localhost:8000/api/v1"

# Shared ablation Vibe set — kept small to limit API costs
ABLATION_VIBES = [
    "A warm, intimate reverb like playing guitar in a wooden cabin",
    "Tape saturation with gritty overdrive",
    "Freezing empty cave reverb with long decay",
    "Glitchy VHS scanline tears with RGB colour split",
    "Soft dreamy blur — gentle Gaussian diffusion",
    "Telephone band-pass filter for voice",
]

print(f"Ablation Vibe set: {len(ABLATION_VIBES)} Vibes.")

## Ablation 1 — AST Hint

Compare generation quality **with** and **without** the ASTBuilder template chain hint injected into the user prompt.

In [ ]:
from generation.dsp.ast_builder import ASTBuilder
from app.config import get_settings
from app.services.prompt_builder import PromptBuilder
from app.models.request_models import GenerateDSPRequest, InputType

settings       = get_settings()
prompt_builder = PromptBuilder(settings)
ast_builder    = ASTBuilder()


def build_dsp_prompt_with_hint(vibe: str) -> tuple[str, str]:
    """Build prompts WITH the AST hint injected."""
    request = GenerateDSPRequest(
        prompt=vibe, input_type=InputType.audio_stream, sample_rate=44100,
        max_parameters=8, force_refresh=True,
    )
    system, user = prompt_builder.build_dsp_prompts(request)
    ast = ast_builder.build(vibe)
    if ast.confidence >= 0.2 and ast.nodes:
        user = f"{user}\n\n{ast.to_faust_hint()}"
    return system, user


def build_dsp_prompt_without_hint(vibe: str) -> tuple[str, str]:
    """Build prompts WITHOUT the AST hint."""
    request = GenerateDSPRequest(
        prompt=vibe, input_type=InputType.audio_stream, sample_rate=44100,
        max_parameters=8, force_refresh=True,
    )
    return prompt_builder.build_dsp_prompts(request)


# Inspect the difference for one Vibe
sample_vibe = ABLATION_VIBES[0]
_, user_with    = build_dsp_prompt_with_hint(sample_vibe)
_, user_without = build_dsp_prompt_without_hint(sample_vibe)

print("=== WITH hint (last 200 chars of user prompt) ===")
print(user_with[-200:])
print("\n=== WITHOUT hint (last 200 chars of user prompt) ===")
print(user_without[-200:])
print(f"\nCharacter delta: +{len(user_with) - len(user_without)} chars with hint")

In [ ]:
# Run both conditions through the live API
# NOTE: This calls the API twice per Vibe — set force_refresh=True to bypass cache

async def generate_dsp(vibe: str) -> dict:
    async with httpx.AsyncClient(timeout=60.0) as client:
        resp = await client.post(
            f"{API_BASE}/generate/dsp",
            json={"prompt": vibe, "input_type": "audio_stream",
                  "sample_rate": 44100, "max_parameters": 8, "force_refresh": True},
        )
        resp.raise_for_status()
        return resp.json()


print("NOTE: AST hint ablation is measured indirectly via compile_time_ms and")
print("self_correction_attempts from the live API (the hint is always injected")
print("server-side). To test the no-hint condition, set confidence threshold to 1.0")
print("in FaustGenerator and re-run the eval_dsp_quality notebook.")
print()

ast_rows = []
for vibe in ABLATION_VIBES:
    ast = ast_builder.build(vibe)
    ast_rows.append({
        "vibe":        vibe[:55],
        "confidence":  ast.confidence,
        "node_count":  len(ast.nodes),
        "chain":       ast.to_faust_hint(),
    })

ast_df = pd.DataFrame(ast_rows)
print("AST confidence scores for ablation Vibes:")
display(ast_df)

In [ ]:
# Visualise AST confidence distribution
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(range(len(ast_df)), ast_df["confidence"], color="steelblue", edgecolor="white")
ax.set_yticks(range(len(ast_df)))
ax.set_yticklabels([v[:40] for v in ast_df["vibe"]], fontsize=9)
ax.axvline(0.2, color="red", linestyle="--", label="Injection threshold (0.2)")
ax.set_xlabel("AST confidence score")
ax.set_title("AST Builder Confidence per Vibe")
ax.legend()
plt.tight_layout()
plt.savefig("ast_confidence.png", dpi=150)
plt.show()

## Ablation 2 — Few-Shot Examples

Compare prompts with and without few-shot examples. Measure token overhead and whether self-correction rates differ.

In [ ]:
from app.services.prompt_builder import _load_few_shots, _format_few_shots, _load_text
from pathlib import Path

few_shots     = _load_few_shots()
dsp_examples  = few_shots.get("dsp", [])
fmt_examples  = _format_few_shots(dsp_examples, max_examples=2)

rows = []
for vibe in ABLATION_VIBES:
    request = GenerateDSPRequest(
        prompt=vibe, input_type=InputType.audio_stream,
        sample_rate=44100, max_parameters=8, force_refresh=True,
    )
    sys_with, usr_with = prompt_builder.build_dsp_prompts(request)

    # Approximate no-few-shot version by removing the example block
    sys_without = sys_with.replace(fmt_examples, "").strip()

    rows.append({
        "vibe":             vibe[:45],
        "tokens_with":      len(sys_with.split()),
        "tokens_without":   len(sys_without.split()),
        "token_overhead":   len(sys_with.split()) - len(sys_without.split()),
        "overhead_pct":     round(
            (len(sys_with.split()) - len(sys_without.split())) / len(sys_without.split()) * 100, 1
        ),
    })

few_shot_df = pd.DataFrame(rows)
print(f"Average few-shot token overhead: {few_shot_df['token_overhead'].mean():.0f} tokens "
      f"({few_shot_df['overhead_pct'].mean():.1f}%)")
display(few_shot_df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Token counts with vs without few shots
x = range(len(few_shot_df))
axes[0].bar(x, few_shot_df["tokens_with"],    label="With few-shots",    alpha=0.8, color="steelblue")
axes[0].bar(x, few_shot_df["tokens_without"], label="Without few-shots", alpha=0.8, color="lightcoral")
axes[0].set_xticks(x)
axes[0].set_xticklabels([v[:20] for v in few_shot_df["vibe"]], rotation=45, ha="right", fontsize=8)
axes[0].set_ylabel("Approx token count")
axes[0].set_title("System Prompt Token Count")
axes[0].legend()

# Overhead percentage
axes[1].bar(x, few_shot_df["overhead_pct"], color="goldenrod", edgecolor="white")
axes[1].set_xticks(x)
axes[1].set_xticklabels([v[:20] for v in few_shot_df["vibe"]], rotation=45, ha="right", fontsize=8)
axes[1].set_ylabel("Token overhead (%)")
axes[1].set_title("Few-Shot Token Overhead")
axes[1].axhline(few_shot_df["overhead_pct"].mean(), color="red", linestyle="--", label="Mean")
axes[1].legend()

plt.tight_layout()
plt.savefig("few_shot_overhead.png", dpi=150)
plt.show()

## Ablation 3 — Cache Similarity Threshold Sweep

Evaluate how the similarity threshold affects cache hit rate for a fixed query set.
Lower threshold → more hits → risk of returning a semantically distant result.
Higher threshold → fewer hits → more fresh generations (higher latency, higher cost).

In [ ]:
from cache.vector_store import VectorStore
from cache.similarity_search import SimilaritySearch
from app.services.cache_service import _simple_embed

# Load the on-disk cache index
store    = VectorStore(index_path="../cache/cache_index.json")
searcher = SimilaritySearch(store, threshold=0.92)

THRESHOLDS = [0.50, 0.60, 0.70, 0.75, 0.80, 0.85, 0.90, 0.92, 0.95, 0.98]

sweep_rows = []
for vibe in ABLATION_VIBES:
    embedding     = _simple_embed(vibe)
    gen_type      = "shader" if any(w in vibe.lower() for w in ["glitch", "blur", "vhs", "colour"]) else "dsp"
    threshold_hits = searcher.sweep_thresholds(embedding, gen_type, THRESHOLDS)
    for thresh, hit in threshold_hits.items():
        sweep_rows.append({"vibe": vibe[:40], "threshold": thresh, "hit": hit, "gen_type": gen_type})

sweep_df = pd.DataFrame(sweep_rows)

# Hit rate at each threshold
hit_rate_by_threshold = sweep_df.groupby("threshold")["hit"].mean() * 100
print("Cache hit rate by threshold:")
display(hit_rate_by_threshold.rename("hit_rate_pct").to_frame())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Line: hit rate vs threshold
axes[0].plot(
    hit_rate_by_threshold.index, hit_rate_by_threshold.values,
    marker="o", color="steelblue", linewidth=2,
)
axes[0].axvline(0.92, color="red", linestyle="--", label="Default threshold (0.92)")
axes[0].set_xlabel("Similarity threshold")
axes[0].set_ylabel("Cache hit rate (%)")
axes[0].set_title("Hit Rate vs Similarity Threshold")
axes[0].set_ylim(0, 105)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Heatmap: hit matrix (Vibe × threshold)
pivot = sweep_df.pivot_table(index="vibe", columns="threshold", values="hit", aggfunc="first")
im = axes[1].imshow(pivot.values.astype(float), aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
axes[1].set_xticks(range(len(pivot.columns)))
axes[1].set_xticklabels([str(t) for t in pivot.columns], rotation=45, ha="right", fontsize=8)
axes[1].set_yticks(range(len(pivot.index)))
axes[1].set_yticklabels([v[:30] for v in pivot.index], fontsize=8)
axes[1].set_title("Cache Hit Heatmap (green=hit, red=miss)")
plt.colorbar(im, ax=axes[1], label="Hit (1) / Miss (0)")

plt.tight_layout()
plt.savefig("cache_threshold_sweep.png", dpi=150)
plt.show()

# Recommendation
optimal = hit_rate_by_threshold[
    hit_rate_by_threshold >= hit_rate_by_threshold.max() * 0.9
].index.max()
print(f"\nRecommended threshold (highest threshold with ≥90% of max hit rate): {optimal}")

## Summary & Recommendations

In [ ]:
print("=" * 60)
print("PROMPT ABLATION SUMMARY")
print("=" * 60)
print()
print("Ablation 1 — AST Hint:")
print(f"  Vibes above injection threshold (≥0.2): "
      f"{(ast_df['confidence'] >= 0.2).sum()}/{len(ast_df)}")
print(f"  Mean confidence: {ast_df['confidence'].mean():.3f}")
print("  Conclusion: Hint injection active for most Vibes. Monitor self-correction")
print("  rate in eval_dsp_quality.ipynb to confirm benefit.")
print()
print("Ablation 2 — Few-Shot Examples:")
print(f"  Mean token overhead: {few_shot_df['token_overhead'].mean():.0f} tokens "
      f"({few_shot_df['overhead_pct'].mean():.1f}%)")
print("  Conclusion: Overhead is manageable. Recommend keeping 2 examples.")
print("  If cost is a concern, reduce to 1 and re-run eval notebooks.")
print()
print("Ablation 3 — Cache Threshold:")
print(f"  Recommended threshold: {optimal}")
print(f"  Max achievable hit rate: {hit_rate_by_threshold.max():.1f}%")
print("  Conclusion: Update settings.cache_similarity_threshold if current")
print("  default (0.92) differs from the recommendation above.")

# Save ablation outputs
ast_df.to_csv("ablation_ast_confidence.csv", index=False)
few_shot_df.to_csv("ablation_few_shot_overhead.csv", index=False)
sweep_df.to_csv("ablation_cache_threshold.csv", index=False)
print("\nAblation CSVs saved.")